# 임베딩 비교 — 4종 (통계피처 / TS2Vec / MOMENT-small / SentenceTransformer)

**실행 전 확인사항**
1. 런타임 → 런타임 유형 변경 → **GPU** 선택 (TS2Vec 필수)
2. Google Drive에 학습데이터-라벨링데이터 폴더 업로드 완료
3. 아래 `DATA_DIR` 경로를 실제 Drive 경로로 수정

In [ ]:
# ── 패키지 설치 ───────────────────────────────────────────────
!pip install -q ts2vec momentfm sentence-transformers xgboost

In [ ]:
# ── Google Drive 마운트 ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# 실제 경로로 수정
DATA_DIR = '/content/drive/MyDrive/학습데이터-라벨링데이터'

In [ ]:
# ── 공통 임포트 ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Protocol

TARGET_HOUSES = [
    'house_067', 'house_049', 'house_054', 'house_011', 'house_017',
    'house_015', 'house_035', 'house_046', 'house_065', 'house_002',
]

In [ ]:
# ── 데이터 로더 ───────────────────────────────────────────────
@dataclass
class HouseDayData:
    house_id: str
    day: date
    profile_1440: np.ndarray
    temperature: float | None
    is_weekday: bool

    def window_kwh(self, start_h: int, end_h: int) -> float:
        return float(self.profile_1440[start_h*60:end_h*60].sum()) / 60.0 / 1000.0


def load_house_data(house_id: str) -> list[HouseDayData]:
    house_path = Path(DATA_DIR) / house_id
    if not house_path.exists():
        print(f'  [경고] {house_id} 없음')
        return []

    # temperature from ch01
    temp_map = {}
    ch01 = house_path / 'ch01.parquet'
    if ch01.exists():
        df = pd.read_parquet(ch01)
        for _, row in df.iterrows():
            t = pd.to_numeric(row.get('temperature', None), errors='coerce')
            if not pd.isna(t):
                temp_map[str(row['date'])] = float(t)

    # build 1440-min profiles from ch02-ch23 events
    day_profiles: dict[date, np.ndarray] = {}
    for ch in range(2, 24):
        p = house_path / f'ch{ch:02d}.parquet'
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
        df['end_time']   = pd.to_datetime(df['end_time'],   errors='coerce')
        df['power_w']    = pd.to_numeric(df['power_consumption'], errors='coerce').fillna(0)
        df = df.dropna(subset=['start_time'])
        for _, row in df.iterrows():
            d = row['start_time'].date()
            if d not in day_profiles:
                day_profiles[d] = np.zeros(1440, dtype=np.float32)
            st = int(row['start_time'].hour * 60 + row['start_time'].minute)
            et_ts = pd.to_datetime(row['end_time'], errors='coerce')
            et = int(et_ts.hour * 60 + et_ts.minute) if pd.notna(et_ts) else st + 1
            st = max(0, min(st, 1439))
            et = max(st+1, min(et, 1440))
            day_profiles[d][st:et] += row['power_w']

    result = []
    for day, prof in sorted(day_profiles.items()):
        temp = temp_map.get(day.strftime('%Y%m%d'))
        is_wd = datetime.combine(day, datetime.min.time()).weekday() < 5
        result.append(HouseDayData(house_id, day, prof, temp, is_wd))
    return result


print('데이터 로딩...')
all_data = {}
for h in TARGET_HOUSES:
    d = load_house_data(h)
    if d:
        all_data[h] = d
        print(f'  {h}: {len(d)}일')
print(f'완료: {len(all_data)}가구')

In [ ]:
# ── 임베더 4종 정의 ───────────────────────────────────────────

# 1. 통계 피처
class StatisticalEmbedder:
    name = '통계피처'
    def embed(self, p, t=None):
        h = p.reshape(24, 60).mean(axis=1)
        total = h.sum() + 1e-9
        nh = h / total
        feat = np.concatenate([h, nh,
                               [h.mean(), h.std(), h.max(), float(np.argmax(h))],
                               [nh[0:6].sum(), nh[6:12].sum(), nh[12:18].sum(), nh[18:24].sum()],
                               [t or 0.0]])
        n = np.linalg.norm(feat)
        return feat / (n + 1e-9)
    def embed_batch(self, ps, ts=None):
        ts = ts or [None]*len(ps)
        return np.stack([self.embed(p, t) for p, t in zip(ps, ts)])


# 2. TS2Vec
class TS2VecEmbedder:
    name = 'TS2Vec'
    def __init__(self):
        from ts2vec import TS2Vec
        self._cls = TS2Vec
        self._model = None
    def fit(self, profiles):
        self._model = self._cls(input_dims=1, output_dims=320)
        self._model.fit(profiles[:, :, np.newaxis], verbose=True)
    def embed(self, p, t=None):
        rep = self._model.encode(p[np.newaxis, :, np.newaxis],
                                  encoding_window='full_series')[0]
        return rep / (np.linalg.norm(rep) + 1e-9)
    def embed_batch(self, ps, ts=None):
        X = np.stack(ps)[:, :, np.newaxis]
        reps = self._model.encode(X, encoding_window='full_series')
        norms = np.linalg.norm(reps, axis=1, keepdims=True)
        return reps / (norms + 1e-9)


# 3. MOMENT-small
class MOMENTEmbedder:
    name = 'MOMENT-small'
    def __init__(self):
        from momentfm import MOMENTForecastingPipeline
        self._model = MOMENTForecastingPipeline.from_pretrained(
            'AutonLab/MOMENT-1-small',
            model_kwargs={'task_name': 'embedding'},
        )
    def embed(self, p, t=None):
        import torch
        p512 = np.interp(np.linspace(0, 1439, 512), np.arange(1440), p)
        x = torch.tensor(p512, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        with torch.no_grad():
            emb = self._model(x).embeddings[0].numpy()
        return emb / (np.linalg.norm(emb) + 1e-9)
    def embed_batch(self, ps, ts=None):
        return np.stack([self.embed(p) for p in ps])


# 4. SentenceTransformer
class SentenceTransformerEmbedder:
    name = 'SentenceTransformer'
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self._model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    def _to_text(self, p, t=None):
        h = p.reshape(24, 60).mean(axis=1)
        parts = [f'{i:02d}시:{w:.1f}W' for i, w in enumerate(h)]
        text = '시간대별 평균 전력 ' + ' '.join(parts)
        if t: text += f' 기온:{t:.1f}도'
        return text
    def embed(self, p, t=None):
        emb = self._model.encode(self._to_text(p, t), normalize_embeddings=True)
        return emb
    def embed_batch(self, ps, ts=None):
        ts = ts or [None]*len(ps)
        return np.stack([self.embed(p, t) for p, t in zip(ps, ts)])


print('임베더 초기화 완료')

In [ ]:
# ── Precision@5 평가 함수 ─────────────────────────────────────
def evaluate_embedder(embedder, all_data, dr_start_h=18, dr_end_h=19, top_k=5, tol=0.3):
    precisions = []
    for house_id, day_data in all_data.items():
        wd = [d for d in day_data if d.is_weekday]
        if len(wd) < top_k + 1:
            continue
        profiles = [d.profile_1440 for d in wd]
        temps    = [d.temperature  for d in wd]

        if hasattr(embedder, 'fit') and embedder._model is None:
            print(f'  [{embedder.name}] fitting on {len(profiles)} profiles...')
            embedder.fit(np.stack(profiles))

        embs = embedder.embed_batch(profiles, temps)

        for i, qd in enumerate(wd):
            q_actual = qd.window_kwh(dr_start_h, dr_end_h)
            if q_actual == 0:
                continue
            others     = np.delete(embs, i, axis=0)
            other_days = [wd[j] for j in range(len(wd)) if j != i]
            sims       = others @ embs[i]
            top_idx    = np.argsort(sims)[-top_k:]
            hits = sum(
                1 for idx in top_idx
                if abs(other_days[idx].window_kwh(dr_start_h, dr_end_h) - q_actual)
                   / (q_actual + 1e-9) <= tol
            )
            precisions.append(hits / top_k)

    return round(float(np.mean(precisions)), 4) if precisions else 0.0, len(precisions)

In [ ]:
# ── 4종 비교 실행 ─────────────────────────────────────────────
embedders = [
    StatisticalEmbedder(),
    SentenceTransformerEmbedder(),
    TS2VecEmbedder(),
    MOMENTEmbedder(),
]

rows = []
for emb in embedders:
    print(f'\n[{emb.name}] 평가 중...')
    try:
        p5, n = evaluate_embedder(emb, all_data)
        rows.append({'임베딩 방법': emb.name, 'Precision@5': p5, '샘플수': n})
        print(f'  -> Precision@5: {p5}')
    except Exception as e:
        rows.append({'임베딩 방법': emb.name, 'Precision@5': 'ERROR', '샘플수': 0})
        print(f'  -> 오류: {e}')

result = pd.DataFrame(rows)
print('\n=== 최종 결과 ===')
print(result.to_string(index=False))

In [ ]:
# ── 시각화 ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Colab 한글 폰트
!apt-get -qq install -y fonts-nanum
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

df = result[result['Precision@5'] != 'ERROR'].copy()
df['Precision@5'] = df['Precision@5'].astype(float)
min_val = df['Precision@5'].min()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E74C3C' if v == df['Precision@5'].max() else '#4A90D9'
          for v in df['Precision@5']]
bars = ax.bar(df['임베딩 방법'], df['Precision@5'], color=colors, width=0.5)
for bar, v in zip(bars, df['Precision@5']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_title('임베딩 방법별 유사 날 검색 정확도 (Precision@5)', fontsize=13)
ax.set_ylabel('Precision@5')
ax.set_ylim(0, max(df['Precision@5']) * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('embedding_compare_colab.png', dpi=150)
plt.show()
print('저장: embedding_compare_colab.png')